# Customer Churn Data

In [1]:
# importing necessary libraries
import pandas as pd
import os

from sklearn.model_selection import train_test_split


# Helper functions

In [2]:
def load_data(file_path):
    data = pd.read_csv(file_path)
    return data
def save_data(data, output_file_path):
    os.makedirs(os.path.dirname(output_file_path), exist_ok=True)

    data.to_csv(output_file_path, index=False)

    parquet_path = output_file_path.replace(".csv", ".parquet") 
    data.to_parquet(parquet_path, index=False, engine="pyarrow")  # preserves datatypes (important for categorical features)


# Loading the data 

In [3]:
data = load_data("../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv")

# Getting to know the data

In [4]:
data.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [5]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [6]:
preprocessed_data = data.copy()

### changing some of the features datatype

In [15]:
# convert total charges to numeric
preprocessed_data['TotalCharges'] = pd.to_numeric(preprocessed_data['TotalCharges'], errors='coerce')
preprocessed_data["SeniorCitizen"] = (preprocessed_data["SeniorCitizen"].map({0: "No", 1: "Yes"}))

In [7]:
preprocessed_data.describe()

,SeniorCitizen,tenure,MonthlyCharges
count,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.761692
std,0.368612,24.559481,30.090047
min,0.000000,0.000000,18.250000
25%,0.000000,9.000000,35.500000
50%,0.000000,29.000000,70.350000
75%,0.000000,55.000000,89.850000
max,1.000000,72.000000,118.750000


# Handling categorical variables

In [8]:

# identify the number of missing values in categorical columns
categorical_cols = preprocessed_data.select_dtypes(include=['object', 'str']).columns
for col in categorical_cols:
    print(f"Missing values in {col}: {preprocessed_data[col].isnull().sum()}")

Missing values in customerID: 0
Missing values in gender: 0
Missing values in Partner: 0
Missing values in Dependents: 0
Missing values in PhoneService: 0
Missing values in MultipleLines: 0
Missing values in InternetService: 0
Missing values in OnlineSecurity: 0
Missing values in OnlineBackup: 0
Missing values in DeviceProtection: 0
Missing values in TechSupport: 0
Missing values in StreamingTV: 0
Missing values in StreamingMovies: 0
Missing values in Contract: 0
Missing values in PaperlessBilling: 0
Missing values in PaymentMethod: 0
Missing values in TotalCharges: 0
Missing values in Churn: 0


In [9]:
# capture unique values in categorical columns
for col in categorical_cols:
    print(f"Unique values in {col}: {preprocessed_data[col].nunique()}")

Unique values in customerID: 7043
Unique values in gender: 2
Unique values in Partner: 2
Unique values in Dependents: 2
Unique values in PhoneService: 2
Unique values in MultipleLines: 3
Unique values in InternetService: 3
Unique values in OnlineSecurity: 3
Unique values in OnlineBackup: 3
Unique values in DeviceProtection: 3
Unique values in TechSupport: 3
Unique values in StreamingTV: 3
Unique values in StreamingMovies: 3
Unique values in Contract: 3
Unique values in PaperlessBilling: 2
Unique values in PaymentMethod: 4
Unique values in TotalCharges: 6531
Unique values in Churn: 2


In [10]:
# drop customerID as it is not a useful feature for prediction
preprocessed_data.drop('customerID', axis=1, inplace=True)
categorical_cols = categorical_cols.drop('customerID')

In [ ]:
# convert categorical columns. For XGBoost, we can convert categorical variables to category dtype in pandas, which allows XGBoost to handle them directly without needing one-hot encoding. This is more efficient and can lead to better performance.
# Since we want to have a shared preprocessing pipeline for all models, we skip this step here and will handle it in the pipeline using OneHotEncoder for all categorical features.
#preprocessed_data[col] = preprocessed_data[col].astype("category")

In [11]:
preprocessed_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            7043 non-null   str    
 1   SeniorCitizen     7043 non-null   int64  
 2   Partner           7043 non-null   str    
 3   Dependents        7043 non-null   str    
 4   tenure            7043 non-null   int64  
 5   PhoneService      7043 non-null   str    
 6   MultipleLines     7043 non-null   str    
 7   InternetService   7043 non-null   str    
 8   OnlineSecurity    7043 non-null   str    
 9   OnlineBackup      7043 non-null   str    
 10  DeviceProtection  7043 non-null   str    
 11  TechSupport       7043 non-null   str    
 12  StreamingTV       7043 non-null   str    
 13  StreamingMovies   7043 non-null   str    
 14  Contract          7043 non-null   str    
 15  PaperlessBilling  7043 non-null   str    
 16  PaymentMethod     7043 non-null   str    
 17  Monthl

In [12]:
# check distribution of target variable and balance if necessary (e.g., using SMOTE or undersampling)
target_col = 'Churn'  # replace with the actual name of your target column
print(f"Distribution of {target_col}:")
print(preprocessed_data[target_col].value_counts())

Distribution of Churn:
Churn
No     5174
Yes    1869
Name: count, dtype: int64


73% No
27% Yes

#### The imbalance is not severe. Therefore we do not need to balance it. We just use ROC AUC, PR AUC, F1, Recall for churn instead of accuracy. We will also use scale_pos_weight = neg / pos in the XGBosst algorithm. 

In [13]:
# target
y = preprocessed_data["Churn"].map({"Yes":1, "No":0})
# features
X = preprocessed_data.drop("Churn", axis=1)

In [14]:
# splite the data into training and testing sets

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.8, random_state=42, stratify=y)
train_df = pd.concat([X_train, y_train], axis=1)
test_df = pd.concat([X_test, y_test], axis=1)

# Saving the processed data

In [15]:
save_data(train_df, "../data/processed/train_data.csv")
save_data(test_df, "../data/processed/test_data.csv")

In [16]:
preprocessed_data.to_csv("../data/processed/preprocessed_WA_Fn-UseC_-Telco-Customer-Churn.csv", index=False)